# Bulk-crystal CXR via the Zhai Monte Carlo pipeline

Coherent X-ray (PXR + CBS) spectra and bremsstrahlung backgrounds from **bulk**
crystals under table-top electron energies, using the segment-sum Monte Carlo
of `cxr_montecarlo.py` (Zhai et al., Nat. Commun. 16, 11218 (2025) architecture):
CASINO-style transport (Browning/Mott elastic + Joy-Luo CSDA) -> per-segment
Eq. (13)/(14) amplitudes with finite-segment sinc$^2$ lineshapes -> per-segment
Beer-Lambert escape absorption -> EDS + aperture detector model.

**Crystals** (all with the relevant axis along the beam, normal incidence, bulk):

| crystal | reflections | geometry notes |
|---|---|---|
| HOPG graphite | $\pm(002), \pm(004)$ | fiber texture: only $(00l)$ coherent -- transport's amorphous assumption is *protected* |
| $\mathrm{MoSe}_2$ | $\pm(002), \pm(004), \pm(006)$ | c-axis $\parallel$ beam; basis verified: 4f Se at $z=0.621$ ($\delta=0.129$, matches the 2.53 $\mathrm{\AA}$ Mo-Se bond) |
| diamond | $\{111\}, \{022\}, \{004\}$ families | beam $\parallel$ [001] zone axis |
| silicon | $\{111\}, \{022\}, \{004\}$ families | beam $\parallel$ [001] zone axis |

**Caveats** (quantified in `kinematic_validity_check.py`):
- Single crystals at an exact zone axis: electron channeling ($\xi_e \sim$ mfp) is
  ignored by the amorphous transport; real measurements tilt a few degrees off axis.
  HOPG is exempt (no coherent transverse potential).
- B-factors approximate (graphite 0.8, $\mathrm{MoSe}_2$ 0.6 $\mathrm{\AA^2}$, unsourced).
- Bremsstrahlung: nonrelativistic Bethe-Heitler Born + Elwert, isotropic emission;
  for Mo/Se ($Z \gtrsim 30$) Seltzer-Berger tables would be better.
- Units are absolute -- Phs/eV/s/nA into the EDS solid
  angle (0.066 sr at $\theta_{obs}=119°$, the Zhai experimental geometry).


In [ ]:
# Project layout: the core library modules live in src/, data in data/.
# Put src/ on the import path so the flat `from cxr_montecarlo import ...` calls
# below resolve. Assumes the notebook is run from the repo root (the same CWD
# the run-checkpoint path assumes).
import sys

sys.path.insert(0, "src")


In [ ]:
import time
from itertools import permutations, product
from tabulate import tabulate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cxr_feranchuk_spence import CRYSTALS, dominant_reflections
from cxr_montecarlo import (
    run_cases,
    beta_from_keV,
    eds_fwhm_eV,
    aperture_fwhm_eV,
    convolve_detector,
    detector_efficiency,
    load_external_brem,
)

# ---- Our detector parameters ------
TIMEPIX3_PIXEL_PITCH = 55e-6
TIMEPIX3_CHIP_WIDTH = 256 * TIMEPIX3_PIXEL_PITCH
TIMEPIX3_ARRAY_WIDTH = 2 * TIMEPIX3_CHIP_WIDTH
TIMEPIX3_DISTANCE = 0.4
TIMEPIX3_DTHETA_OBS_RAD = 2 * np.atan((TIMEPIX3_CHIP_WIDTH / 2) / TIMEPIX3_DISTANCE)
TIMEPIX3_DOMEGA_SR = TIMEPIX3_CHIP_WIDTH**2 / TIMEPIX3_DISTANCE**2

# ---- experiment DEFAULTS: every entry can be overridden per crystal ------------
# (just add the same key to a config in the next cell)
DEFAULTS = dict(
    theta_obs_deg=119.0,  # detector polar angle from the beam [deg]
    dtheta_obs_deg=16.6,  # detector polar-angle span (line smearing) [deg]
    domega_sr=0.066,  # detector solid angle [sr]
    thickness_ang=1470.0,  # sample thickness [Ang]; 1e7 = 1 mm (bulk)
    tilt_deg=-30.0,  # sample-normal polar tilt [deg]
    tilt_azim_deg=0.0,
    energies_kev=(
        17.5,
        20.0,
        22.5,
        25.0,
    ),  # beam energies [keV]  # tilt azimuth [deg]: 0 = in beam-detector plane (pitch)
    # entrance face toward the detector. Must be nonzero
    # (negative) at theta_obs = 90: an untilted infinite
    # slab absorbs photons traveling parallel to its faces
)

# beam energies live in DEFAULTS now (energies_kev) so each crystal can
# override them -- e.g. SEM energies for HOPG, 200 keV for TEM MoSe2
NE = 450  # electrons per transport run (lines)
NE_BREM = 100  # electrons for the background (smooth, needs fewer)
PER_NA = 6.2415e9  # electrons/s at 1 nA
BEAM_CURRENT_NA = 5  # nA
CONVOLVE_WITH_DET = False
APPLY_DETECTOR_QE = True  # polymer-window + Al + grid losses (detector_efficiency)
# plotted Phs/eV/s/nA appear to be as-detected, so apply when comparing to the paper

BREM_SOURCE = "mc"  # "mc": our Born+Elwert MC | "external": per-config
#   brem_file (e.g. a NIST DTSA-II export -- the tool Zhai et al. used for
#   their backgrounds) | "none": pure background-subtracted view


def detected_background(r):
    """Background in DETECTED units (Phs/eV/s/nA) on r['E_grid'].

    'mc'      : our Monte Carlo background, with window QE and detector
                convolution applied per the flags above;
    'external': r['case']['brem_file'] loaded via load_external_brem and
                treated as already-detected (no QE/convolution re-applied);
    'none'    : zeros.
    """
    E = r["E_grid"]
    if BREM_SOURCE == "none":
        return np.zeros_like(E)
    if BREM_SOURCE == "external":
        path = r["case"].get("brem_file")
        return load_external_brem(path, E) if path else np.zeros_like(E)
    qe = detector_efficiency(E) if APPLY_DETECTOR_QE else 1.0
    b = r["brem"] * qe
    if CONVOLVE_WITH_DET:
        b = convolve_detector(E, b, r["fwhm"])
    return b * r["scale"]


def fmt_thickness(t_ang):
    """Compact human thickness label from Angstroms: 316A / 31.6nm / 17um / 1mm."""
    if t_ang < 1e2:
        return f"{t_ang:g}A"
    if t_ang < 1e4:
        return f"{t_ang / 10:g}nm"
    if t_ang < 1e7:
        return f"{t_ang / 1e4:g}um"
    return f"{t_ang / 1e7:g}mm"


def family(hkl):
    """All sign/permutation members of a cubic reflection family."""
    out = set()
    for perm in permutations(hkl):
        for signs in product((1, -1), repeat=3):
            out.add(tuple(p * s for p, s in zip(perm, signs)))
    return sorted(out)


def hex_family(hkl):
    """
    All members of a HEXAGONAL reflection family: the six-fold star about
    the c-axis ((h, k) -> (-k, h+k)) combined with +/-l. (The cubic
    family() helper is wrong for hexagonal indices -- it permutes l with h.)
    """
    h, k, l = hkl
    out = set()
    for _ in range(6):
        h, k = -k, h + k
        for ll in (l, -l):
            out.add((h, k, ll))
    return sorted(out)


def pm(*hkls):
    """A list of reflections together with their negatives."""
    out = []
    for h in hkls:
        out += [tuple(h), tuple(-x for x in h)]
    return out


def n_of(crystal, element):
    """Number density [1/Ang^3] of one element in a crystal's unit cell."""
    info = CRYSTALS[crystal]
    count = sum(1 for el, _ in info["basis"] if el == element)
    return count / info["V_cell"]


mose2_general_params = dict(
    crystal="mose2",
    composition=[("Mo", n_of("mose2", "Mo")), ("Se", n_of("mose2", "Se"))],
    # automatic top-|chi_g| families (Zhai Table 5 methodology): for our
    # corrected MoSe2 structure this picks {002}, {100}, {103}, {006} --
    # {006} outranks their {004} because the z_Se fix doubled S(006).
    # Crank n_families to include more (each family costs compute).
    # NOTE: B_ang2 is a single isotropic value applied to all reflections.
    # This does not reflect reality (in reality, anisotropic Debye-Waller factor)
    hkl_list=dominant_reflections("mose2", n_families=4, B_ang2=0.6),
    beam_uvw=(0, 0, 2),
    B_ang2=0.6,
    # E_grid=np.arange(800.0, 1200.0, 1.0),
    E_grid=np.arange(500.0, 1750.0, 3.0),
)
diamond_general_params = dict(
    crystal="diamond",
    composition=[("C", n_of("diamond", "C"))],
    # auto-selected: {111}, {022}, {113}, {004} (= Zhai Table 5 exactly)
    hkl_list=dominant_reflections("diamond", n_families=4, B_ang2=0.21),
    beam_uvw=(4, 0, 0),
    B_ang2=0.21,
    E_grid=np.arange(100.0, 5000.0, 2.0),
)
silicon_general_params = dict(
    crystal="silicon",
    composition=[("Si", n_of("silicon", "Si"))],
    # auto-selected: {111}, {022}, {113}, {004} (= Zhai Table 5 exactly)
    hkl_list=dominant_reflections("silicon", n_families=4, B_ang2=0.46),
    beam_uvw=(4, 4, 0),
    B_ang2=0.46,
    E_grid=np.arange(100.0, 5000.0, 3.0),
)
graphite_general_params = dict(
    crystal="graphite",
    composition=[("C", n_of("graphite", "C"))],
    # HOPG is FIBER-TEXTURED: only the (00l) c-axis reflections are coherent
    # across grains -- do NOT use dominant_reflections here (its {100}/{101}
    # families are real for single-crystal graphite but wash out in HOPG).
    hkl_list=pm((0, 0, 2), (0, 0, 4)),
    # Zhai geometry: beam along the c-axis [001] (the film/HOPG normal),
    # then the per-sample Table 4 tilts. [110] is an IN-PLANE direction --
    # the (110) in their SI is only the CBED thickness-measurement reflection.
    beam_uvw=(0, 0, 1),
    B_ang2=0.8,
    # E_grid=np.arange(600.0, 1200.0, 1.0),
    E_grid=np.arange(100.0, 5000.0, 3.0),
)


c:\venvs\sci\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


Using GPU


In [ ]:
# ---- per-crystal configuration ------------------------------------------------
# Required keys: crystal, composition, hkl_list, B_ang2, E_grid.
# Optional keys override DEFAULTS for that crystal only:
#   thickness_ang, beam_uvw, tilt_deg, theta_obs_deg, dtheta_obs_deg, domega_sr

crystal_configs = {}

# # ---- HOPG in the SEM: beam energies inherited from DEFAULTS (17.5-25 keV) ------
# zhai_graphite_params = [
#     (9.5, 40, 17e4),
#     (9.5, 40, 500e4),
#     (10, 40, 1e7),
# ]  # [(10, 0, 17e4), (10, 0, 500e4), (10, 0, 1e7)]
# for pol, azim, thickness in zhai_graphite_params:
#     crystal_configs[f"HOPG {fmt_thickness(thickness)} pol:{pol:g} az:{azim:g}"] = {
#         **graphite_general_params,
#         "tilt_deg": float(pol),
#         "tilt_azim_deg": float(azim),
#         "thickness_ang": float(thickness),
#     }

# # ---- MoSe2 in the TEM (SI Fig. 5c): 200 keV, 147 nm film, tilt scan ------------
# # TEM EDS geometry: Zhai's group characterized the SAME JEOL 2010HR in
# # Huang et al., Adv. Sci. 2022 (arXiv:2203.14765): theta_obs ~ 112.5 deg,
# # dtheta_obs ~ 12 deg, EDS resolution ~97 eV. dOmega is not quoted, but
# # Zhai's own SEM pair (16.6 deg, 0.066 sr) is exactly a circular cone of
# # half-angle dtheta/2, so the same convention gives 2*pi*(1-cos 6 deg)
# # = 0.0344 sr. 200 keV caveat: Browning total elastic cross section is
# # extrapolated beyond 30 keV (gamma-corrected amplitudes are in).
# # NEGATIVE tilt (entrance face toward the detector): this is the sign that
# # reproduces SI Fig. 5c -- flux RISES with tilt magnitude and peaks sit at
# # ~990-1040 eV. Positive tilt walks the PXR lobe away from the detector
# # (falling trend, x10-60 weaker by 20 deg).
# mose2_params = [(-angle, 0, 1470) for angle in [10, 15, 17.5, 20]]
# for pol, azim, thickness in mose2_params:
#     crystal_configs[f"MoSe2 {fmt_thickness(thickness)} pol:{pol:g} az:{azim:g}"] = {
#         **mose2_general_params,
#         "tilt_deg": float(pol),
#         "tilt_azim_deg": float(azim),
#         "thickness_ang": float(thickness),
#         "energies_kev": (200.0,),
#         "theta_obs_deg": 112.5,
#         "dtheta_obs_deg": 12.0,
#         "domega_sr": 0.0344,
#     }

# pol_angles = np.linspace(2, 88, 10)
# azim_angles = np.linspace(2, 88, 10)
pol_angles = np.linspace(-36, -1, 14, endpoint=True)
azim_angles = np.linspace(-9, 9, 7, endpoint=True)
energies = (30, 45, 60)
thicknesses = [
    2e4,
]
our_mose2_params = []

for pol_angle in pol_angles:
    for azim_angle in azim_angles:
        for thickness in thicknesses:
            our_mose2_params.append((pol_angle, azim_angle, thickness))

for pol, azim, thickness in our_mose2_params:
    crystal_configs[f"$MoSe_2$ {fmt_thickness(thickness)} pol:{pol:g} az:{azim:g}"] = {
        **mose2_general_params,
        "tilt_deg": pol,
        "tilt_azim_deg": azim,
        "thickness_ang": thickness,
        "energies_kev": energies,
        "theta_obs_deg": 90,
        "dtheta_obs_deg": np.degrees(TIMEPIX3_DTHETA_OBS_RAD),
        "domega_sr": TIMEPIX3_DOMEGA_SR,
    }

# ---- build the case list (DEFAULTS merged with per-crystal overrides) ----------
cases = []
geo_rows = []
for i_c, (name, cfg) in enumerate(crystal_configs.items()):
    p = {**DEFAULTS, **cfg}
    for i_e, E0 in enumerate(p["energies_kev"]):
        cases.append(
            dict(
                name=name,
                crystal=p["crystal"],
                composition=p["composition"],
                hkl_list=p["hkl_list"],
                B_ang2=p["B_ang2"],
                E0_keV=float(E0),
                thickness_ang=p["thickness_ang"],
                E_grid=(
                    p["E_grid"][0],
                    p["E_grid"][-1] + (p["E_grid"][1] - p["E_grid"][0]),
                    p["E_grid"][1] - p["E_grid"][0],
                ),
                theta_obs_rad=np.deg2rad(p["theta_obs_deg"]),
                tilt_deg=p["tilt_deg"],
                tilt_azim_deg=p.get("tilt_azim_deg", 0.0),
                brem_file=p.get("brem_file"),
                beam_uvw=p["beam_uvw"],
                Ne=NE,
                Ne_brem=NE_BREM,
                seed=1000 * i_c + 10 * i_e + 1,
                # used downstream (detector model / unit scaling), ignored by run_case:
                dtheta_obs_rad=np.deg2rad(p["dtheta_obs_deg"]),
                domega_sr=p["domega_sr"],
            )
        )
    geo_rows.append(
        [
            name,
            len(p["hkl_list"]),
            p["thickness_ang"] / 1e4,
            str(p["beam_uvw"]),
            p["tilt_deg"],
            p.get("tilt_azim_deg", 0.0),
            str(list(p["energies_kev"])),
            p["theta_obs_deg"],
            p["dtheta_obs_deg"],
            p["domega_sr"],
        ]
    )

print(
    tabulate(
        geo_rows,
        headers=[
            "config",
            "refl",
            "t [um]",
            "beam uvw",
            "tilt [deg]",
            "tilt azim [deg]",
            "energies [keV]",
            "theta_obs [deg]",
            "dtheta [deg]",
            "dOmega [sr]",
        ],
        tablefmt="github",
        floatfmt=(None, None, ".3f", None, "+.1f", ".1f", None, ".1f", ".1f", "g"),
    )
)


| config                               |   refl |   t [um] | beam uvw   |   tilt [deg] |   tilt azim [deg] | energies [keV]   |   theta_obs [deg] |   dtheta [deg] |   dOmega [sr] |
|--------------------------------------|--------|----------|------------|--------------|-------------------|------------------|-------------------|----------------|---------------|
| $MoSe_2$ 2um pol:-40 az:0            |     22 |    2.000 | (0, 0, 2)  |        -40.0 |               0.0 | [30, 45, 60]     |                90 |            2.0 |    0.00123904 |
| $MoSe_2$ 2um pol:-40 az:5.45455      |     22 |    2.000 | (0, 0, 2)  |        -40.0 |               5.5 | [30, 45, 60]     |                90 |            2.0 |    0.00123904 |
| $MoSe_2$ 2um pol:-40 az:10.9091      |     22 |    2.000 | (0, 0, 2)  |        -40.0 |              10.9 | [30, 45, 60]     |                90 |            2.0 |    0.00123904 |
| $MoSe_2$ 2um pol:-40 az:16.3636      |     22 |    2.000 | (0, 0, 2)  |        -40.0 |       

In [3]:
# ---- post-processing + plotting helpers (defined before the run) ---------------
# `results` is filled incrementally by the run cell as each case finishes, so it
# survives a kernel interrupt -- stop a long scan and still plot what completed.
results = {}  # name -> {E0 -> result dict}

colors = ["r", "y", "g", "b", "m", "c"]
mode = "EDS-convolved" if CONVOLVE_WITH_DET else "intrinsic"


def store_result(case, out):
    """Post-process one finished Monte Carlo case into `results`."""
    name, E0 = case["name"], case["E0_keV"]
    E_grid = out["E_grid"]
    E_pk = E_grid[np.argmax(out["spec"])]
    fwhm = np.sqrt(
        eds_fwhm_eV(E_pk) ** 2
        + aperture_fwhm_eV(
            E_pk, beta_from_keV(E0), case["theta_obs_rad"], case["dtheta_obs_rad"]
        )
        ** 2
    )
    results.setdefault(name, {})[E0] = dict(
        E_grid=E_grid,
        spec=out["spec"],
        brem=out["brem"],
        E_pk=E_pk,
        fwhm=fwhm,
        eta=out["eta"],
        scale=case["domega_sr"] * PER_NA,  # (per e per sr) -> (per s per nA)
        case=case,
    )


def panel_title(name, case):
    """Geometry summary for one config (the key carries t / tilt already)."""
    uvws = case["beam_uvw"]
    uvw_str = f"[{uvws[0]},{uvws[1]},{uvws[2]}]"
    mat = name.split(" ")[0]
    polar_tilt = case["tilt_deg"]
    azim_tilt = case["tilt_azim_deg"]
    thickness = case["thickness_ang"]
    theta_obs_deg = np.degrees(case["theta_obs_rad"])
    dtheta_obs_deg = np.degrees(case["dtheta_obs_rad"])
    return (
        f"\n{mat}, {thickness / 1e4:0.1f}"
        + f" $\\mu m, \\theta_\\mathrm{{tilt}}={polar_tilt:g}\\degree$, "
        f"$\\phi_\\mathrm{{tilt}}={azim_tilt:g}\\degree$" + "\n" + f"uvw {uvw_str}, "
        f"$\\theta_o={theta_obs_deg:g}\\degree$, $\\Delta\\theta_o={dtheta_obs_deg:g}\\degree$, "
        f"$d\\Omega={case['domega_sr'] * 1e3:3.0f}$ msr"
    )


def plot_config(ax, name, include_brem):
    """Draw one config's panel (all its beam energies) onto `ax`."""
    last_r = None
    for i, E0 in enumerate(sorted(results[name])):
        r = results[name][E0]
        last_r = r
        c = colors[i % len(colors)]
        qe = detector_efficiency(r["E_grid"]) if APPLY_DETECTOR_QE else 1.0
        line_in = r["spec"] * qe
        line_det = (
            convolve_detector(r["E_grid"], line_in, r["fwhm"])
            if CONVOLVE_WITH_DET
            else line_in
        )
        brem_det = detected_background(r) / r["scale"]  # honors BREM_SOURCE
        if include_brem:
            i_pk = np.argmax(line_det)
            E_pk = r["E_grid"][i_pk]
            ax.plot(
                r["E_grid"] / 1e3,
                line_det * r["scale"],
                color=c,
                ls="-",
                lw=1.0,
                label=f"{(str(E0) + ' keV, '):>10s}peak {E_pk:g} eV",
            )
            ax.vlines(E_pk / 1e3, 0, line_det[i_pk] * r["scale"], color=c, ls="-.")
        else:
            i_pk = np.argmax(line_det + brem_det)
            E_pk = r["E_grid"][i_pk]
            ax.plot(
                r["E_grid"] / 1e3,
                (line_det + brem_det) * r["scale"],
                color=c,
                ls="-",
                lw=1.0,
                label=f"{E0:0.1f} keV, peak {E_pk:g} eV",
            )
            ax.plot(r["E_grid"] / 1e3, brem_det * r["scale"], color=c, ls="--", lw=1.0)
            ax.vlines(
                E_pk / 1e3,
                0,
                (line_det[i_pk] + brem_det[i_pk]) * r["scale"],
                color=c,
                ls="-.",
            )
    ax.set_title(panel_title(name, last_r["case"]), fontsize=12)
    ax.set_xlabel("Photon energy (keV)", fontsize=12)
    ax.set_ylabel("Intensity (Phs/eV/s/nA)", fontsize=12)
    ax.set_ylim(bottom=0)
    ax.margins(x=0)
    ax.grid(alpha=0.3)
    ax.legend(title="solid: line\ndashed: brem" if include_brem else None, fontsize=11)


def plot_grid(names, include_brem, ncols=2, title=None):
    """One figure with a panel per config name (skips names not yet in results)."""
    names = [n for n in names if n in results]
    if not names:
        return None
    nrows = (len(names) + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False
    )
    for ax in axes.ravel()[len(names) :]:
        ax.axis("off")
    for ax, name in zip(axes.ravel(), names):
        plot_config(ax, name, include_brem)
    if title:
        fig.suptitle(title, fontsize=15)
    fig.tight_layout()
    return fig


In [4]:
# ---- re-grouped plotting: one figure per energy, panels by tilt, curves by azimuth


def plot_tilt_panel(ax, group_results, include_brem):
    """One panel: fixed (energy, tilt_deg), one curve per tilt_azim_deg."""
    sorted_results = sorted(group_results, key=lambda r: r["case"]["tilt_azim_deg"])
    for i, r in enumerate(sorted_results):
        c = colors[i % len(colors)]
        azim = r["case"]["tilt_azim_deg"]
        qe = detector_efficiency(r["E_grid"]) if APPLY_DETECTOR_QE else 1.0
        line_in = r["spec"] * qe
        line_det = (
            convolve_detector(r["E_grid"], line_in, r["fwhm"])
            if CONVOLVE_WITH_DET
            else line_in
        )
        brem_det = detected_background(r) / r["scale"]
        if include_brem:
            ax.plot(
                r["E_grid"] / 1e3,
                (line_det + brem_det) * r["scale"],
                color=c,
                ls="-",
                lw=1.0,
                label=f"$\\phi_{{tilt}}={azim:.1f}\\degree$",
            )
            ax.plot(r["E_grid"] / 1e3, brem_det * r["scale"], color=c, ls="--", lw=0.7)
        else:
            ax.plot(
                r["E_grid"] / 1e3,
                line_det * r["scale"],
                color=c,
                ls="-",
                lw=1.0,
                label=f"$\\phi_{{tilt}}={azim:.1f}\\degree$",
            )
    case = sorted_results[0]["case"]
    mat = case["name"].split()[0]
    ax.set_title(
        f"{mat}, {case['thickness_ang'] / 1e4:.1f} $\\mu$m, {case['E0_keV']:g} keV, "
        f"$\\theta_{{tilt}}={case['tilt_deg']:g}\\degree$",
        fontsize=11,
    )
    ax.set_xlabel("Photon energy (keV)", fontsize=10)
    ax.set_ylabel("Intensity (Phs/eV/s/nA)", fontsize=10)
    ax.set_ylim(bottom=0)
    ax.margins(x=0)
    ax.grid(alpha=0.3)
    ax.legend(
        title="solid: total\ndashed: brem" if include_brem else None,
        fontsize=9,
    )


def plot_by_energy(include_brem, ncols=5, title_prefix=None):
    """One figure per beam energy. Panels = tilt_deg, curves = tilt_azim_deg."""
    all_r = [results[name][E0] for name in results for E0 in results[name]]
    if not all_r:
        return []
    energies = sorted(set(r["case"]["E0_keV"] for r in all_r))
    tilt_degs = sorted(set(r["case"]["tilt_deg"] for r in all_r))
    figs = []
    for E0 in energies:
        panels = []
        for tilt in tilt_degs:
            group = [
                r
                for r in all_r
                if r["case"]["E0_keV"] == E0 and r["case"]["tilt_deg"] == tilt
            ]
            if group:
                panels.append(group)
        if not panels:
            continue
        nrows = (len(panels) + ncols - 1) // ncols
        fig, axes = plt.subplots(
            nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False
        )
        for ax in axes.ravel()[len(panels) :]:
            ax.axis("off")
        for ax, group in zip(axes.ravel(), panels):
            plot_tilt_panel(ax, group, include_brem)
        fig.suptitle(f"{E0:g} keV — {title_prefix or mode}", fontsize=15)
        fig.tight_layout()
        figs.append(fig)
    return figs

In [ ]:
# ---- photon-counting statistics table (per chunk, reused for the full roll-up)
# Factored out of the old end-of-notebook cell so the run cell can print the
# stats for each finished chunk right under its plot, instead of one giant table
# at the very end.
from IPython.display import display  # explicit so it also works headless


def summary_table(names, current_na=BEAM_CURRENT_NA):
    """Photon-counting stats for the given configs: line peak position, the
    EDS-convolved peak height, peak-over-background, and integrated line / brem /
    total count rates [counts/s] at `current_na`. Returns a rounded DataFrame
    (empty if none of `names` are in `results` yet)."""
    rows = []
    for name in names:
        for E0 in sorted(results.get(name, {})):
            r = results[name][E0]
            qe = detector_efficiency(r["E_grid"]) if APPLY_DETECTOR_QE else 1.0
            line_det = convolve_detector(r["E_grid"], r["spec"] * qe, r["fwhm"])
            brem_det = detected_background(r) / r["scale"]  # honors BREM_SOURCE
            i_pk = np.argmax(line_det)
            line_cts = np.trapezoid(r["spec"], r["E_grid"]) * r["scale"] * current_na
            brem_cts = np.trapezoid(r["brem"], r["E_grid"]) * r["scale"] * current_na
            rows.append(
                {
                    "config": name,
                    "Ee [keV]": E0,
                    "line [eV]": r["E_grid"][i_pk],
                    "peak [Phs/eV/s]": line_det[i_pk] * r["scale"] * current_na,
                    "peak/bg": line_det[i_pk] / brem_det[i_pk],
                    "line [cts/s]": line_cts,
                    "brem [cts/s]": brem_cts,
                    "total [cts/s]": line_cts + brem_cts,
                }
            )
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.round(
        {
            "line [eV]": 0,
            "peak [Phs/eV/s]": 2,
            "peak/bg": 2,
            "line [cts/s]": 1,
            "brem [cts/s]": 1,
            "total [cts/s]": 1,
        }
    )


def show_summary(names, current_na=BEAM_CURRENT_NA):
    """Render the photon-counting stats table for `names` (e.g. one chunk)."""
    df = summary_table(names, current_na)
    if df.empty:
        return
    print(
        ("window-QE applied, " if APPLY_DETECTOR_QE else "unity QE, ")
        + f"beam current {current_na:g} nA  |  "
        "peak/bg = EDS-convolved peak height / background at the peak"
    )
    display(df)


In [ ]:
# ---- run + stream plots, chunk by chunk ----------------------------------------
# One worker pool for the whole scan (no repeated spawn cost). Cases stream into
# `results` as they finish; once every beam energy of PANELS_PER_CHUNK configs is
# in, that chunk is plotted AND its photon-counting table is printed -- the first
# figures + numbers appear in minutes, not at the very end. Interrupt the kernel
# anytime: drawn panels and `results` persist.
#
# RESUME reloads an on-disk checkpoint and skips cases already computed, so a
# crashed/restarted kernel resumes where it left off. Set RESUME=False (or delete
# CHECKPOINT_PATH) to force a clean run.
import os
import pickle

from cxr_montecarlo import _GPU

PANELS_PER_CHUNK = 4
# Worker count. With a GPU, the heavy spectrum work runs on the one device, so
# spawning N processes just makes N CUDA contexts fight over it -- that crawls
# the desktop and multiplies VRAM use, with no speedup (transport is only ~20%
# of a case). Run serially in-process (0). Without a GPU, fan out across cores
# (Ryzen 5900X has 12); None lets run_cases pick ~3/4 of the logical CPUs.
MAX_WORKERS = 0 if _GPU else 10
RESUME = True
CHECKPOINT_PATH = "zhai_run_checkpoint.pkl"

if RESUME and os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "rb") as f:
        results.update(pickle.load(f))
    print(
        f"resumed {sum(len(v) for v in results.values())} cases from {CHECKPOINT_PATH}"
    )

todo = [
    c for c in cases if not (c["name"] in results and c["E0_keV"] in results[c["name"]])
]
print(
    f"{len(todo)} of {len(cases)} cases to run "
    f"({len(cases) - len(todo)} already cached)"
)

# how many cases (beam energies) each not-yet-done config still owes
remaining = {}
for c in todo:
    remaining[c["name"]] = remaining.get(c["name"], 0) + 1
done_unplotted = []


def on_case_done(i, case, out):
    store_result(case, out)
    name = case["name"]
    remaining[name] -= 1
    if remaining[name] == 0:  # every beam energy of this config is in
        with open(CHECKPOINT_PATH, "wb") as f:  # crash-safe at config granularity
            pickle.dump(results, f)
        done_unplotted.append(name)
        if len(done_unplotted) >= PANELS_PER_CHUNK:
            batch = done_unplotted[:PANELS_PER_CHUNK]
            del done_unplotted[:PANELS_PER_CHUNK]
            plot_grid(batch, include_brem=True, title=f"chunk ({mode})")
            plt.show()
            show_summary(batch)  # photon-counting stats for just this chunk


t_all = time.perf_counter()
run_cases(todo, max_workers=MAX_WORKERS, callback=on_case_done)
if done_unplotted:  # trailing configs that didn't fill a full chunk
    plot_grid(done_unplotted, include_brem=True, title=f"final chunk ({mode})")
    plt.show()
    show_summary(done_unplotted)  # stats for the trailing chunk
worker_desc = "serial, GPU" if MAX_WORKERS == 0 else f"<= {MAX_WORKERS} workers"
print(f"{len(todo)} cases in {time.perf_counter() - t_all:.0f} s ({worker_desc})")


In [ ]:
# ---- re-plot by energy (one figure per beam energy, panels by tilt_deg) --------
# Each panel shows curves for every tilt_azim_deg at a fixed (energy, tilt_deg).
plot_by_energy(
    include_brem=True,
    title_prefix=f"CXR + bremsstrahlung ({mode})",
)
plot_by_energy(
    include_brem=False,
    title_prefix=f"coherent lines only, background removed ({mode})",
)
plt.show()

In [ ]:
# ---- complete photon-counting roll-up (all configs) ----------------------------
# Per-chunk tables now print under each chunk during the run (cell above). This
# is the full table across every config -- handy after a RESUME, where cached
# configs don't stream a chunk of their own. Same numbers, just not split up.
show_summary(list(results))


## Timepix3 detector response

The plots above are the *intrinsic* emitted spectra (with the Zhai EDS window/resolution model). This section instead pushes them through a forward model of the actual **2×2 Timepix3 (Si sensor)** — photoabsorption → charge generation (Fano) → charge sharing (diffusion + 3×3 erf split) → per-pixel threshold (counting) → ToT energy noise → Poisson statistics (`timepix_response.py`).

The counting threshold (~525 e⁻ ≈ **1.9 keV**) sits *above* much of the MoSe₂ signal, so expect strong sub-2 keV suppression, a near-threshold turn-on, and a charge-loss low-energy tail on every line.

> **⚠ Hardware placeholders:** sensor thickness (300 µm Si) and bias (100 V) are guesses. Set `TPX_THICKNESS_UM` / `TPX_BIAS_V` in the next cell to the real quad values — the charge-sharing width (and so the tail and the turn-on sharpness) scales as 1/√(bias), the single most sensitive knob.


In [ ]:
# ---- Timepix3 forward model: setup + efficiency/resolution -----------------
import importlib
import timepix_response as tpx

importlib.reload(tpx)  # pick up edits to the module without a kernel restart

# !!! PLACEHOLDERS -- set to the real quad values. sigma_diff ~ 1/sqrt(bias),
#     so bias is the most sensitive knob for charge sharing / the low-E tail.
TPX_THICKNESS_UM = 300.0  ### FILL IN -- Si sensor thickness [um]
TPX_BIAS_V = 100.0  ### FILL IN -- applied bias [V]
TPX_NMC = 80000  # MC photons per input energy
TPX_SEED = 0
TPX_HW = dict(thickness_um=TPX_THICKNESS_UM, bias_v=TPX_BIAS_V)
E_thr_keV = tpx.THRESHOLD_E * tpx.W_EHP_EV / 1e3  # counting threshold [keV]

# efficiency + resolution: evaluate on a smooth grid (independent of the spectra)
E_eff = np.arange(200.0, 8000.0, 100.0)
_resp = tpx.build_response(
    E_eff, np.arange(0.0, 9800.0, 25.0), n_mc=TPX_NMC, seed=TPX_SEED, **TPX_HW
)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.6))
# -- left: efficiencies --
axL.plot(
    E_eff / 1e3,
    _resp["eps_abs"],
    "k--",
    lw=1.2,
    label=r"$\epsilon_\mathrm{abs}$ (Si absorption)",
)
axL.plot(
    E_eff / 1e3,
    _resp["P_det"],
    "b-",
    lw=1.8,
    label=r"$P_\mathrm{det}$ (abs $\times$ counting)",
)
axL.axvline(E_thr_keV, color="r", ls=":", label=f"threshold = {E_thr_keV:.2f} keV")
axL.set(
    xlabel="Photon energy (keV)",
    ylabel="efficiency",
    ylim=(0, 1.05),
    title=f"Detection efficiency ({TPX_THICKNESS_UM:g} $\\mu$m Si, "
    f"{TPX_BIAS_V:g} V, $\\sigma_\\mathrm{{diff}}$="
    f"{_resp['sigma_diff_um']:.1f} $\\mu$m)",
)
axL.margins(x=0)
axL.grid(alpha=0.3)
axL.legend()
# -- right: energy resolution + charge-loss bias --
axR.plot(
    E_eff / 1e3, tpx.energy_fwhm_eV(E_eff), "k-", lw=1.5, label="analytic, single-pixel"
)
axR.plot(
    E_eff / 1e3,
    _resp["fwhm_rec"],
    "b.",
    ms=4,
    label="MC effective (tail + multi-pixel)",
)
axR.set(
    xlabel="Photon energy (keV)",
    ylabel="energy FWHM (eV)",
    title="Energy resolution & charge-loss bias",
)
axR.margins(x=0)
axR.grid(alpha=0.3)
axR.legend(loc="upper left")
axR2 = axR.twinx()
axR2.plot(E_eff / 1e3, 100 * (1 - _resp["mean_rec"] / E_eff), "g-", lw=1, alpha=0.5)
axR2.set_ylabel("mean charge-loss deficit (%)", color="g")
axR2.tick_params(axis="y", colors="g")
axR2.set_ylim(bottom=0)
fig.tight_layout()
plt.show()


In [ ]:
# ---- Timepix3 detected spectra (per-grid response, cached & reused) --------
# Responses are cached by energy grid + hardware (tpx.get_response), so each
# distinct grid present in `results` -- e.g. a stale-checkpoint mix of 2500 and
# 3000 eV runs -- builds its matrix once and every case on that grid reuses it
# (a cheap matmul). NaN/inf spectra (e.g. a bad-theta case) -> zero detected.
print(
    f"sigma_diff = {tpx.sigma_diffusion_um(**TPX_HW):.1f} um "
    f"({TPX_THICKNESS_UM:g} um Si, {TPX_BIAS_V:g} V)"
)


def detected_total(r):
    """Incident and detected (line + brem) [Phs/eV/s/nA] on r['E_grid']."""
    incident = (r["spec"] + r["brem"]) * r["scale"]
    resp = tpx.get_response(r["E_grid"], n_mc=TPX_NMC, seed=TPX_SEED, **TPX_HW)
    return incident, resp.apply(incident)


def plot_detected_panel(ax, group):
    """One panel: incident (dotted) vs Timepix3-detected (solid) per azimuth."""
    sr = sorted(group, key=lambda r: r["case"]["tilt_azim_deg"])
    ymax = 0.0
    for i, r in enumerate(sr):
        c = colors[i % len(colors)]
        inc, det = detected_total(r)
        finite = inc[np.isfinite(inc)]
        if finite.size:
            ymax = max(ymax, float(finite.max()))
        az = r["case"]["tilt_azim_deg"]
        ax.plot(r["E_grid"] / 1e3, inc, color=c, ls=":", lw=0.7, alpha=0.45)
        ax.plot(
            r["E_grid"] / 1e3,
            det,
            color=c,
            ls="-",
            lw=1.2,
            label=f"$\\phi_{{tilt}}={az:.1f}\\degree$",
        )
    case = sr[0]["case"]
    ax.axvline(E_thr_keV, color="0.4", ls=":", lw=0.8)
    if ymax > 0:
        ax.set_yscale("log")
        ax.set_ylim(ymax * 1e-6, ymax * 2)
    ax.set_title(
        f"{case['name'].split()[0]}, {case['E0_keV']:g} keV, "
        f"$\\theta_{{tilt}}={case['tilt_deg']:g}\\degree$",
        fontsize=11,
    )
    ax.set_xlabel("Photon energy (keV)", fontsize=10)
    ax.set_ylabel("Phs/eV/s/nA", fontsize=10)
    ax.margins(x=0)
    ax.grid(alpha=0.3, which="both")
    ax.legend(title="solid: detected\ndotted: incident", fontsize=8)


def plot_detected_by_energy(ncols=5):
    """One figure per beam energy; panels = tilt_deg, curves = tilt_azim_deg."""
    all_r = [results[n][E0] for n in results for E0 in results[n]]
    energies = sorted({r["case"]["E0_keV"] for r in all_r})
    tilts = sorted({r["case"]["tilt_deg"] for r in all_r})
    for E0 in energies:
        panels = [
            g
            for g in (
                [
                    r
                    for r in all_r
                    if r["case"]["E0_keV"] == E0 and r["case"]["tilt_deg"] == t
                ]
                for t in tilts
            )
            if g
        ]
        nrows = (len(panels) + ncols - 1) // ncols
        fig, axes = plt.subplots(
            nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False
        )
        for ax in axes.ravel()[len(panels) :]:
            ax.axis("off")
        for ax, g in zip(axes.ravel(), panels):
            plot_detected_panel(ax, g)
        fig.suptitle(
            f"{E0:g} keV — Timepix3 detected (solid) vs incident (dotted)", fontsize=15
        )
        fig.tight_layout()
    plt.show()


plot_detected_by_energy()


In [ ]:
# ---- Timepix3 Poisson 'measured' realization ------------------------------
# One representative config per beam energy (the highest detected rate); draw
# Poisson counts over TPX_INTEGRATION_S at BEAM_CURRENT_NA -- what an actual
# acquisition records, statistical noise and all.
TPX_INTEGRATION_S = 600.0  # acquisition live time [s]
_rng = np.random.default_rng(TPX_SEED)

_all = [results[n][E0] for n in results for E0 in results[n]]
_energies = sorted({r["case"]["E0_keV"] for r in _all})
fig, axes = plt.subplots(
    1, len(_energies), figsize=(6 * len(_energies), 4.6), squeeze=False
)
for ax, E0 in zip(axes.ravel(), _energies):
    grp = [r for r in _all if r["case"]["E0_keV"] == E0]
    r = max(grp, key=lambda rr: np.trapezoid(detected_total(rr)[1], rr["E_grid"]))
    _, det = detected_total(r)
    counts, expected = tpx.poisson_counts(
        r["E_grid"], det * BEAM_CURRENT_NA, TPX_INTEGRATION_S, _rng
    )
    ax.step(
        r["E_grid"] / 1e3,
        counts,
        where="mid",
        color="k",
        lw=0.7,
        label=f"measured ({TPX_INTEGRATION_S:g} s @ {BEAM_CURRENT_NA:g} nA)",
    )
    ax.plot(r["E_grid"] / 1e3, expected, "r-", lw=1.3, label="expected mean")
    ax.axvline(E_thr_keV, color="b", ls=":", lw=0.8, label="threshold")
    ax.set_title(
        f"{E0:g} keV, $\\theta_{{tilt}}={r['case']['tilt_deg']:g}\\degree$, "
        f"$\\phi={r['case']['tilt_azim_deg']:g}\\degree$  "
        f"({counts.sum():.0f} cts)",
        fontsize=10,
    )
    ax.set_xlabel("Photon energy (keV)")
    ax.set_ylabel("counts / bin")
    ax.margins(x=0)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle(
    f"Timepix3 Poisson 'measured' spectra "
    f"({TPX_THICKNESS_UM:g} $\\mu$m Si, {TPX_BIAS_V:g} V)",
    fontsize=14,
)
fig.tight_layout()
plt.show()
